# Engagement 2: Mobile-Money Agent Data Cleaning

## Task 10: Initial Data Audit

The purpose of this task is to inspect the agent register and transaction records, identify data quality issues, and assess the condition of the datasets before cleaning.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(44)
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(
        ["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20
    ),
    "onboarded": pd.date_range(
        "2024-01-01", periods=20, freq="MS"
    ).strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n
    ),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (
        pd.to_datetime("2026-03-01") +
        pd.to_timedelta(rng.integers(0, 60, n), unit="D")
    ).strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = (
    txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
)

txns = pd.concat(
    [txns, txns.sample(12, random_state=1)],
    ignore_index=True
)

txns.loc[
    rng.choice(len(txns), 8, replace=False), "agent_id"
] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("Agent and transaction datasets generated successfully.")
print("Agents:", agents.shape)
print("Transactions:", txns.shape)

Agent and transaction datasets generated successfully.
Agents: (20, 3)
Transactions: (612, 5)


## Initial Data Quality Audit

The agent register contains 20 records, while the transaction dataset contains 612 records. The transaction dataset requires checks for duplicate transactions, missing and invalid amounts, inconsistent transaction types, date-format inconsistencies and transactions linked to unknown agents.

In [ ]:
# Inspect both datasets
print("AGENT REGISTER")
display(agents.head())
print(agents.info())
print("\nMissing values:")
display(agents.isnull().sum())

print("\nTRANSACTION DATA")
display(txns.head())
print(txns.info())
print("\nMissing values:")
display(txns.isnull().sum())

# Identify data quality issues
audit = pd.DataFrame([
    {
        "issue": "Duplicate transaction IDs",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows_affected": txns["txn_id"].duplicated().sum()
    },
    {
        "issue": "Missing transaction amounts",
        "dimension": "Completeness",
        "column": "amount",
        "rows_affected": txns["amount"].isna().sum()
    },
    {
        "issue": "Invalid transaction types",
        "dimension": "Consistency",
        "column": "txn_type",
        "rows_affected": (~txns["txn_type"].astype(str).str.strip().str.lower().isin(
            ["deposit", "withdrawal"]
        )).sum()
    },
    {
        "issue": "Non-numeric transaction amounts",
        "dimension": "Validity",
        "column": "amount",
        "rows_affected": pd.to_numeric(txns["amount"], errors="coerce").isna().sum()
    },
    {
        "issue": "Inconsistent date formats",
        "dimension": "Validity",
        "column": "txn_date",
        "rows_affected": txns["txn_date"].astype(str).str.contains("/").sum()
    },
    {
        "issue": "Unknown agent IDs",
        "dimension": "Referential integrity",
        "column": "agent_id",
        "rows_affected": (~txns["agent_id"].isin(agents["agent_id"])).sum()
    }
])

print("\nDATA QUALITY AUDIT")
display(audit)

AGENT REGISTER


,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   agent_id   20 non-null     object
 1   county     20 non-null     object
 2   onboarded  20 non-null     object
dtypes: object(3)
memory usage: 612.0+ bytes
None

Missing values:


,0
agent_id,0
county,0
onboarded,0



TRANSACTION DATA


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   txn_id    612 non-null    object
 1   agent_id  612 non-null    object
 2   txn_type  612 non-null    object
 3   amount    597 non-null    object
 4   txn_date  612 non-null    object
dtypes: object(5)
memory usage: 24.0+ KB
None

Missing values:


,0
txn_id,0
agent_id,0
txn_type,0
amount,15
txn_date,0



DATA QUALITY AUDIT


,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Missing transaction amounts,Completeness,amount,15
2,Invalid transaction types,Consistency,txn_type,0
3,Non-numeric transaction amounts,Validity,amount,22
4,Inconsistent date formats,Validity,txn_date,5
5,Unknown agent IDs,Referential integrity,agent_id,8


In [ ]:
display(audit)

,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Missing transaction amounts,Completeness,amount,15
2,Invalid transaction types,Consistency,txn_type,0
3,Non-numeric transaction amounts,Validity,amount,22
4,Inconsistent date formats,Validity,txn_date,5
5,Unknown agent IDs,Referential integrity,agent_id,8


## Task 11: Duplicate Transaction Removal

The transaction dataset contained 12 duplicate transaction IDs. Duplicate records can distort transaction counts and financial summaries. The duplicates were removed using the transaction ID as the unique identifier, retaining the first occurrence of each transaction.

In [ ]:
# Remove duplicate transaction IDs
txns_clean = txns.drop_duplicates(
    subset="txn_id",
    keep="first"
).copy()

print("Original transaction records:", len(txns))
print("Clean transaction records:", len(txns_clean))
print("Duplicates removed:", len(txns) - len(txns_clean))
print("Remaining duplicate IDs:", txns_clean["txn_id"].duplicated().sum())

Original transaction records: 612
Clean transaction records: 600
Duplicates removed: 12
Remaining duplicate IDs: 0


## Task 12: Standardization of Transaction Types

The transaction type column contained inconsistent capitalization and spacing. These inconsistencies were standardized by removing extra spaces and applying consistent title case. The final categories are Deposit and Withdrawal.

In [ ]:
# Standardize transaction types
txns_clean["txn_type"] = (
    txns_clean["txn_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "deposit": "Deposit",
        "withdrawal": "Withdrawal"
    })
)

print("Transaction type counts:")
display(txns_clean["txn_type"].value_counts(dropna=False))

print("\nUnique transaction types:")
print(txns_clean["txn_type"].unique())

Transaction type counts:


,count
txn_type,
Deposit,373
Withdrawal,227



Unique transaction types:
['Deposit' 'Withdrawal']


## Task 13: Data Type Conversion

The transaction amount column was stored as text, while the transaction date column was also stored as text. The amount values were converted to numeric format, and transaction dates were converted to datetime format. Invalid values were identified for further treatment rather than silently discarded.

In [ ]:
# Convert transaction amounts to numeric
txns_clean["amount"] = pd.to_numeric(
    txns_clean["amount"],
    errors="coerce"
)

# Convert transaction dates to datetime
txns_clean["txn_date"] = pd.to_datetime(
    txns_clean["txn_date"],
    format="mixed",
    errors="coerce"
)

print("Data types after conversion:")
print(txns_clean.dtypes)

print("\nMissing amounts:", txns_clean["amount"].isna().sum())
print("Invalid dates:", txns_clean["txn_date"].isna().sum())

print("\nDate range:")
print(txns_clean["txn_date"].min(), "to", txns_clean["txn_date"].max())

Data types after conversion:
txn_id              object
agent_id            object
txn_type            object
amount             float64
txn_date    datetime64[ns]
dtype: object

Missing amounts: 20
Invalid dates: 0

Date range:
2026-03-01 00:00:00 to 2026-04-29 00:00:00


## Task 14: Missing Value Treatment

After removing duplicate records and converting transaction amounts to numeric format, 20 missing transaction amounts were identified. These values were imputed using the median transaction amount. A new Boolean column, amount_imputed, was created to distinguish original amounts from imputed values. This approach retains transaction records while reducing the influence of extreme values on the imputation.

In [ ]:
# Identify records with missing amounts
txns_clean["amount_imputed"] = txns_clean["amount"].isna()

# Calculate median transaction amount
median_amount = txns_clean["amount"].median()

# Impute missing amounts
txns_clean["amount"] = txns_clean["amount"].fillna(median_amount)

print("Median used for imputation:", median_amount)
print("Amounts imputed:", txns_clean["amount_imputed"].sum())
print("Missing amounts remaining:", txns_clean["amount"].isna().sum())

display(txns_clean.head())

Median used for imputation: 1776.5
Amounts imputed: 20
Missing amounts remaining: 0


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed
0,TX00001,AG009,Deposit,353.0,2026-03-03,False
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False
3,TX00004,AG009,Deposit,652.0,2026-04-24,False
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False


## Task 15: Outlier Detection and Treatment

The Interquartile Range (IQR) method was used to identify unusually high and low transaction amounts. Transactions exceeding the upper IQR boundary were flagged for investigation. The KES 250,000 transactions require verification because they may represent legitimate high-value transactions rather than errors. Outliers will be retained but flagged pending confirmation to avoid removing valid financial records.

In [ ]:
# Calculate IQR boundaries
Q1 = txns_clean["amount"].quantile(0.25)
Q3 = txns_clean["amount"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Flag outliers
txns_clean["is_outlier"] = (
    (txns_clean["amount"] < lower_bound) |
    (txns_clean["amount"] > upper_bound)
)

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower boundary:", lower_bound)
print("Upper boundary:", upper_bound)

print("\nNumber of outliers:", txns_clean["is_outlier"].sum())

print("\nKES 250,000 transactions:")
display(txns_clean[txns_clean["amount"] == 250000])

print("\nOutlier transaction records:")
display(txns_clean[txns_clean["is_outlier"]].head(10))

Q1: 1105.0
Q3: 2868.5
IQR: 1763.5
Lower boundary: -1540.25
Upper boundary: 5513.75

Number of outliers: 53

KES 250,000 transactions:


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier
306,TX00307,AG012,Deposit,250000.0,2026-03-02,False,True
315,TX00316,AG014,Deposit,250000.0,2026-04-01,False,True
424,TX00425,AG005,Withdrawal,250000.0,2026-03-02,False,True
463,TX00464,AG009,Withdrawal,250000.0,2026-04-11,False,True
594,TX00595,AG016,Deposit,250000.0,2026-03-11,False,True



Outlier transaction records:


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True
45,TX00046,AG016,Deposit,5632.0,2026-03-02,False,True
55,TX00056,AG099,Deposit,8507.0,2026-04-01,False,True
56,TX00057,AG017,Deposit,6006.0,2026-03-05,False,True
57,TX00058,AG011,Withdrawal,8035.0,2026-04-16,False,True
63,TX00064,AG007,Withdrawal,6566.0,2026-04-20,False,True
65,TX00066,AG016,Withdrawal,7116.0,2026-04-22,False,True
67,TX00068,AG019,Withdrawal,7962.0,2026-03-03,False,True
69,TX00070,AG005,Deposit,5856.0,2026-04-11,False,True
87,TX00088,AG014,Deposit,7072.0,2026-04-27,False,True


## Task 16: Merging Transactions with the Agent Register

The cleaned transaction records were merged with the agent register using agent_id as the common key. This adds county information to each transaction and allows county-level analysis. Transactions linked to unknown agent IDs are retained and assigned an "Unknown" county rather than being deleted, preserving transaction records for further investigation.

In [ ]:
# Merge cleaned transactions with agent information
merged_txns = txns_clean.merge(
    agents[["agent_id", "county"]],
    on="agent_id",
    how="left",
    indicator=True
)

# Identify unknown agent records
unknown_agents = merged_txns[merged_txns["_merge"] == "left_only"]

print("Transactions after merge:", len(merged_txns))
print("Matched transactions:", (merged_txns["_merge"] == "both").sum())
print("Unknown agent transactions:", len(unknown_agents))

# Preserve unknown records and label their county
merged_txns["county"] = merged_txns["county"].fillna("Unknown")

# Remove the merge indicator
merged_txns.drop(columns=["_merge"], inplace=True)

print("\nCounty values:")
print(merged_txns["county"].value_counts())

display(merged_txns.head())

Transactions after merge: 600
Matched transactions: 593
Unknown agent transactions: 7

County values:
county
Nairobi    180
Mombasa    133
Kisumu     111
Nakuru      99
Kiambu      70
Unknown      7
Name: count, dtype: int64


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru


## Task 17: County-Level Analysis

The cleaned transactions were analyzed by county and transaction type. The analysis calculates total transaction value, number of transactions and average transaction value for deposits and withdrawals. A weekly pivot table was also created to compare transaction values across counties.

In [ ]:
# County summary by transaction type
county_summary = (
    merged_txns.groupby(["county", "txn_type"])
    .agg(
        Total_Value=("amount", "sum"),
        Number_of_Transactions=("txn_id", "count"),
        Average_Value=("amount", "mean")
    )
    .reset_index()
)

print("COUNTY-LEVEL TRANSACTION SUMMARY")
display(county_summary)

# Create weekly transaction period
merged_txns["week"] = (
    merged_txns["txn_date"]
    .dt.to_period("W")
    .astype(str)
)

# Weekly transaction value by county
weekly_pivot = pd.pivot_table(
    merged_txns,
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

print("\nWEEKLY TRANSACTION VALUE BY COUNTY")
display(weekly_pivot)

COUNTY-LEVEL TRANSACTION SUMMARY


,county,txn_type,Total_Value,Number_of_Transactions,Average_Value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77050.5,27,2853.722222
2,Kisumu,Deposit,436420.5,74,5897.574324
3,Kisumu,Withdrawal,76374.5,37,2064.175676
4,Mombasa,Deposit,148986.0,72,2069.250000
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543590.0,118,4606.694915
7,Nairobi,Withdrawal,162036.0,62,2613.483871
8,Nakuru,Deposit,184890.0,60,3081.500000
9,Nakuru,Withdrawal,328594.5,39,8425.500000



WEEKLY TRANSACTION VALUE BY COUNTY


week,2026-02-23/2026-03-01,2026-03-02/2026-03-08,2026-03-09/2026-03-15,2026-03-16/2026-03-22,2026-03-23/2026-03-29,2026-03-30/2026-04-05,2026-04-06/2026-04-12,2026-04-13/2026-04-19,2026-04-20/2026-04-26,2026-04-27/2026-05-03
county,,,,,,,,,,
Kiambu,0.0,18317.0,267771.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,287933.0,23186.0,28125.0,27077.0,41077.0,15341.0,29416.0,37439.0,21308.0
Mombasa,6759.0,285669.5,30477.5,25144.5,17050.0,38344.0,47092.5,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,52860.5,43107.5,41373.5,282931.0,56786.5,56432.5,65187.0,30934.5
Nakuru,2743.5,56574.0,28730.0,39795.0,18998.5,18145.0,276609.0,26152.5,30862.0,14875.0
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0


In [ ]:

%%writefile cleaning_pipeline.py

import pandas as pd


def clean_agent_data(txns, agents):
    """
    Clean mobile-money transactions and merge agent information.

    Returns:
        clean_df: cleaned transaction DataFrame
        log: DataFrame recording data quality issues and affected rows
    """

    clean_df = txns.copy()
    log_entries = []

    # 1. Remove duplicate transactions
    duplicates = clean_df["txn_id"].duplicated().sum()

    log_entries.append({
        "issue": "Duplicate transactions removed",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows_affected": int(duplicates)
    })

    clean_df = clean_df.drop_duplicates(
        subset="txn_id",
        keep="first"
    ).copy()

    # 2. Standardize transaction types
    original_types = clean_df["txn_type"].copy()

    clean_df["txn_type"] = (
        clean_df["txn_type"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "deposit": "Deposit",
            "withdrawal": "Withdrawal"
        })
    )

    invalid_types = clean_df["txn_type"].isna().sum()

    log_entries.append({
        "issue": "Invalid transaction types",
        "dimension": "Consistency",
        "column": "txn_type",
        "rows_affected": int(invalid_types)
    })

    # 3. Convert amounts to numeric
    clean_df["amount"] = pd.to_numeric(
        clean_df["amount"],
        errors="coerce"
    )

    clean_df["amount_imputed"] = clean_df["amount"].isna()

    missing_amounts = clean_df["amount_imputed"].sum()

    log_entries.append({
        "issue": "Missing or invalid amounts",
        "dimension": "Completeness",
        "column": "amount",
        "rows_affected": int(missing_amounts)
    })

    median_amount = clean_df["amount"].median()

    clean_df["amount"] = clean_df["amount"].fillna(
        median_amount
    )

    # 4. Convert transaction dates
    clean_df["txn_date"] = pd.to_datetime(
        clean_df["txn_date"],
        format="mixed",
        errors="coerce"
    )

    invalid_dates = clean_df["txn_date"].isna().sum()

    log_entries.append({
        "issue": "Invalid transaction dates",
        "dimension": "Validity",
        "column": "txn_date",
        "rows_affected": int(invalid_dates)
    })

    # 5. Detect outliers using IQR
    q1 = clean_df["amount"].quantile(0.25)
    q3 = clean_df["amount"].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    clean_df["is_outlier"] = (
        (clean_df["amount"] < lower_bound) |
        (clean_df["amount"] > upper_bound)
    )

    outlier_count = clean_df["is_outlier"].sum()

    log_entries.append({
        "issue": "Outliers flagged",
        "dimension": "Accuracy",
        "column": "amount",
        "rows_affected": int(outlier_count)
    })

    # 6. Merge agent register
    clean_df = clean_df.merge(
        agents[["agent_id", "county"]],
        on="agent_id",
        how="left"
    )

    unknown_agents = clean_df["county"].isna().sum()

    log_entries.append({
        "issue": "Unknown agent IDs",
        "dimension": "Referential integrity",
        "column": "agent_id",
        "rows_affected": int(unknown_agents)
    })

    clean_df["county"] = clean_df["county"].fillna("Unknown")

    # 7. Create cleaning log
    log = pd.DataFrame(log_entries)

    return clean_df, log


Overwriting cleaning_pipeline.py


In [ ]:
from cleaning_pipeline import clean_agent_data

final_clean_df, cleaning_log = clean_agent_data(txns, agents)

print("FINAL CLEANED DATA")
print("Rows:", len(final_clean_df))
print("Columns:", final_clean_df.shape[1])

print("\nCLEANING LOG")
display(cleaning_log)

print("\nFINAL DATA SAMPLE")
display(final_clean_df.head())

FINAL CLEANED DATA
Rows: 600
Columns: 8

CLEANING LOG


,issue,dimension,column,rows_affected
0,Duplicate transactions removed,Uniqueness,txn_id,12
1,Invalid transaction types,Consistency,txn_type,0
2,Missing or invalid amounts,Completeness,amount,20
3,Invalid transaction dates,Validity,txn_date,0
4,Outliers flagged,Accuracy,amount,53
5,Unknown agent IDs,Referential integrity,agent_id,7



FINAL DATA SAMPLE


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru


## Task 19: Export of Cleaned Transaction Data

The cleaned transaction dataset was exported for further analysis and reporting. The final dataset contains standardized transaction types, numeric transaction amounts, valid transaction dates, imputation flags, outlier flags and agent county information. Unknown agents were retained and labelled as Unknown to preserve transaction records.

In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

final_clean_df.to_csv(
    "outputs/agent_transactions_clean.csv",
    index=False
)

print("Cleaned data exported successfully.")
print("Final rows:", len(final_clean_df))
print("Duplicate transaction IDs:",
      final_clean_df["txn_id"].duplicated().sum())
print("Missing amounts:", final_clean_df["amount"].isna().sum())
print("Missing dates:", final_clean_df["txn_date"].isna().sum())
print("Transaction types:",
      final_clean_df["txn_type"].unique())
print("Unknown county records:",
      (final_clean_df["county"] == "Unknown").sum())

Cleaned data exported successfully.
Final rows: 600
Duplicate transaction IDs: 0
Missing amounts: 0
Missing dates: 0
Transaction types: ['Deposit' 'Withdrawal']
Unknown county records: 7


In [ ]:
print("FINAL SUBMISSION CHECK")
print("----------------------")
print("Rows:", len(final_clean_df))
print("Columns:", final_clean_df.shape[1])
print("Duplicate IDs:", final_clean_df["txn_id"].duplicated().sum())
print("Missing amounts:", final_clean_df["amount"].isna().sum())
print("Missing dates:", final_clean_df["txn_date"].isna().sum())
print("Unknown agents:", (final_clean_df["county"] == "Unknown").sum())
print("Outliers flagged:", final_clean_df["is_outlier"].sum())
print("Imputed amounts:", final_clean_df["amount_imputed"].sum())

FINAL SUBMISSION CHECK
----------------------
Rows: 600
Columns: 8
Duplicate IDs: 0
Missing amounts: 0
Missing dates: 0
Unknown agents: 7
Outliers flagged: 53
Imputed amounts: 20


In [ ]:
%%writefile requirements.txt
pandas
numpy

Writing requirements.txt


In [ ]:
from google.colab import files

files.download("outputs/agent_transactions_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# County summary by transaction type
county_summary = (
    final_clean_df.groupby(["county", "txn_type"])
    .agg(
        Total_Value=("amount", "sum"),
        Number_of_Transactions=("txn_id", "count"),
        Average_Value=("amount", "mean")
    )
    .reset_index()
)

display(county_summary)

# Weekly transaction value by county
final_clean_df["week"] = (
    final_clean_df["txn_date"].dt.to_period("W").astype(str)
)

weekly_pivot = pd.pivot_table(
    final_clean_df,
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

display(weekly_pivot)

,county,txn_type,Total_Value,Number_of_Transactions,Average_Value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77050.5,27,2853.722222
2,Kisumu,Deposit,436420.5,74,5897.574324
3,Kisumu,Withdrawal,76374.5,37,2064.175676
4,Mombasa,Deposit,148986.0,72,2069.250000
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543590.0,118,4606.694915
7,Nairobi,Withdrawal,162036.0,62,2613.483871
8,Nakuru,Deposit,184890.0,60,3081.500000
9,Nakuru,Withdrawal,328594.5,39,8425.500000


week,2026-02-23/2026-03-01,2026-03-02/2026-03-08,2026-03-09/2026-03-15,2026-03-16/2026-03-22,2026-03-23/2026-03-29,2026-03-30/2026-04-05,2026-04-06/2026-04-12,2026-04-13/2026-04-19,2026-04-20/2026-04-26,2026-04-27/2026-05-03
county,,,,,,,,,,
Kiambu,0.0,18317.0,267771.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,287933.0,23186.0,28125.0,27077.0,41077.0,15341.0,29416.0,37439.0,21308.0
Mombasa,6759.0,285669.5,30477.5,25144.5,17050.0,38344.0,47092.5,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,52860.5,43107.5,41373.5,282931.0,56786.5,56432.5,65187.0,30934.5
Nakuru,2743.5,56574.0,28730.0,39795.0,18998.5,18145.0,276609.0,26152.5,30862.0,14875.0
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0


In [19]:
county_summary = (
    final_clean_df.groupby(["county", "txn_type"])
    .agg(
        Total_Value=("amount", "sum"),
        Number_of_Transactions=("txn_id", "count"),
        Average_Value=("amount", "mean")
    )
    .reset_index()
)

display(county_summary)

,county,txn_type,Total_Value,Number_of_Transactions,Average_Value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77050.5,27,2853.722222
2,Kisumu,Deposit,436420.5,74,5897.574324
3,Kisumu,Withdrawal,76374.5,37,2064.175676
4,Mombasa,Deposit,148986.0,72,2069.250000
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543590.0,118,4606.694915
7,Nairobi,Withdrawal,162036.0,62,2613.483871
8,Nakuru,Deposit,184890.0,60,3081.500000
9,Nakuru,Withdrawal,328594.5,39,8425.500000


In [21]:
final_clean_df["week"] = (
    final_clean_df["txn_date"].dt.to_period("W").astype(str)
)

weekly_pivot = pd.pivot_table(
    final_clean_df,
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

display(weekly_pivot)

week,2026-02-23/2026-03-01,2026-03-02/2026-03-08,2026-03-09/2026-03-15,2026-03-16/2026-03-22,2026-03-23/2026-03-29,2026-03-30/2026-04-05,2026-04-06/2026-04-12,2026-04-13/2026-04-19,2026-04-20/2026-04-26,2026-04-27/2026-05-03
county,,,,,,,,,,
Kiambu,0.0,18317.0,267771.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,287933.0,23186.0,28125.0,27077.0,41077.0,15341.0,29416.0,37439.0,21308.0
Mombasa,6759.0,285669.5,30477.5,25144.5,17050.0,38344.0,47092.5,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,52860.5,43107.5,41373.5,282931.0,56786.5,56432.5,65187.0,30934.5
Nakuru,2743.5,56574.0,28730.0,39795.0,18998.5,18145.0,276609.0,26152.5,30862.0,14875.0
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0
